Transformer 

In [ ]:
#add imports

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

from tensorflow.keras.datasets import imdb

#The dataset has 25 thousand words but we only used 10 thousand for the example
(x_train,y_train), (x_test,y_test) = imdb.load_data(num_words=10000)


#config for transformer
embed_dim = 256
latent_dim = 2048
num_heads = 8
vocab_size = 3000
sequence_length = 200
batch_size = 64
maxlen = 200
#we make all the sequences have the same length
x_train = pad_sequences(x_train, maxlen = maxlen)
x_test = pad_sequences(x_test, maxlen = maxlen)

In [5]:
#First we create the Positional Embedding, which is essential in a transformer to know the context
class PositinalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)

        #first we tokenize the input that comes in (attribute configuration)
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,output_dim=embed_dim
        )
        #then we convert the token into a vector of numbers (embedding) (attribute configuration)
        self.positions_embedding = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        #we define the class attributes and assign the passed parameters...
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_positions = self.positions_embedding(positions)
        embedded_tokens = self.token_embedding(inputs)
        embedded_positions = self.positions_embedding(positions)
        return embedded_positions + embedded_tokens
    
    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim
            }
        )
        return config
    
class EncoderTranformer(layers.Layer):
    def __init__(self,embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        #var definitions
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads

        # Attention
        self.attention = layers.MultiHeadAttention(
            num_heads = num_heads, key_dim = embed_dim
        )

        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim,activation="relu"),
                layers.Dense(embed_dim),
            ]

        )
        self.layersnorm_1 = layers.LayerNormalization()
        self.layersnorm_2 = layers.LayerNormalization()
        self.support_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = np.cast(mask[:,None,:],dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=padding_mask
        )

        proj_input = self.layersnorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layersnorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config



In [6]:
#we create a keras input that will receive an integer (int32) without a specific size
encoder_inputs = keras.Input(shape=(None,), dtype="int32",name="encoder_inputs")
#then we perform the embedding and positional embedding of the tokens and I pass the keras input that we created before
x = PositinalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = EncoderTranformer(embed_dim,latent_dim,num_heads)(x)

#we add a layer to reduce the dimensionality of the encoder output
x = layers.GlobalAveragePooling1D()(encoder_outputs)

# dense layer with binary output between (0,1)
outputs = layers.Dense(1, activation="sigmoid")(x)

Transformer = keras.Model(
    encoder_inputs,
    outputs,
    name="Transformer",
)
